# Practice Session 08: Data streams

In this session we will take a large corpus of documents and compute some statistics using data streams methods.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

⭐ **Regarding the usage of generative AI.** You may use generative AI to (a) understand how to do certain operations, and (b) generate short snippets of code, that you can incorporate into your practice after reviewing them. In that case, add a comment `#AI` before the snippet. You can also add `#AI (edited)` if you edited the code before incorporating it.

⚠️ It is forbidden to use generative AI in these practices for answering the text-based questions.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

Author: <font color="#2680EB">Your name here</font>

E-mail: <font color="#2680EB">Your e-mail here</font>

AI used: <font color="#2680EB">Maker and model, or "none".</font>

Date: <font color="#2680EB">The current date here</font>

In [ ]:
import io
import random
import statistics
import secrets
import re
import gzip

# Natural language processing
import nltk

# Timer
from timeit import default_timer as timer

# 0. Dataset and how to iterate

The input file contain the text of [The Odyssey](https://www.gutenberg.org/ebooks/1727) by Homer, an epic written 2,700-2,800 years ago, and it traces the return to Ithaca of its king, Odysseus, after the Trojan War.

During this practice, **we will never load this file in memory.**

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE THIS CODE AS-IS

INPUT_FILE = "odyssey.txt.gz"

The function `read_by_words` is a [generator](https://wiki.python.org/moin/Generators), that is, a function that behaves as an iterator. This is a common pattern used in stream processing, and in Python is implemented with the `yield` keyword, instead of `return`.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE THIS CODE AS-IS

POS_NOUN = 'NN'
POS_ADJECTIVE = 'JJ'
POS_VERB = 'VB'
POS_ADVERB = 'RB'
POS_DETERMINER = 'DT'

# Producer in Python that reads a file by words that are nouns
def read_by_parts_of_speech(filename, parts_of_speech, max_words=-1, report_every=-1, use_pos_prefix=True):
    
    # Default value for report_every
    if report_every < 0 and max_words > 0:
        report_every = int(max_words/5)
    
    # Open the input file
    with gzip.open(INPUT_FILE, "rt", encoding='utf8') as file:
        
        # Initialize counter of words to stop at max_words
        counter = 0
    
        # Iterate through lines in the file
        for line in file:
                                        
            if counter > max_words and max_words != -1:
                print(f"Stopping: read {counter} words")
                break
                
            for sentence in nltk.sent_tokenize(line):
                
                words = []
                if parts_of_speech is None or len(parts_of_speech) == 0:
                    words = nltk.word_tokenize(sentence)
                else:
                    tagged = nltk.pos_tag(nltk.word_tokenize(sentence))
                    if use_pos_prefix:
                        words = [part[0] for part in tagged if any(part[1].startswith(prefix) for prefix in parts_of_speech)]
                    else:
                        words = [part[0] for part in tagged if part[1] in parts_of_speech]

                for word in words:

                    # Ensure the word is alphabetic (no punctuation, no numbers), and has at least two characters
                    if re.match(r'^[a-zA-Z]{2,}$', word):
                    
                        counter += 1

                        # Report
                        if (report_every != -1) and (counter % report_every == 0):
                            if max_words == -1:
                                print(f"- Read {counter} words so far")
                            else:
                                print(f"- Read {counter}/{max_words} words so far")

                        # Produce the word in lowercase
                        yield word.lower()

        print(f"Stopping: end of the input file reached")

We will do a first pass over the data. Here we will read only the first 30K nouns. Try with a larger limit if your computer is fast, with a lower limit if your computer is slow. Find something that makes one pass take about 30 seconds and use it for development.

**Tip:** NLTK may complain that you have some missing files. The following commands may help:

```python3
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
```

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE THIS CODE AS-IS,
# BUT if it takes more than 30 seconds to run, then reduce MAX_WORDS parameter below

# Maximum number of words to read, one pass should take less than 30 seconds in your computer
MAX_WORDS = 400000

# Parts of speech we will focus on
POS_SELECTED = [POS_NOUN, POS_ADJECTIVE, POS_VERB, POS_ADVERB, POS_DETERMINER]

# We use prefixes because there are many variants (e.g., NN is noun, NNS is singular noun, and so on)

print(f"Reading up to {MAX_WORDS} words from {INPUT_FILE}")
start = timer()
for word in read_by_parts_of_speech(INPUT_FILE, POS_SELECTED, max_words=MAX_WORDS, use_pos_prefix=True):
    # Prints 1/1000 of words
    if random.random() < 0.001:
        print("-- Current word '%s'" % (word)) 

end = timer()
print(f"Ellapsed time: {end-start:.2f} seconds")

# 1. Determine approximately the top-10 words

Instead of loading the entire dataset in main memory, we will use reservoir sampling to determine approximately the top-10 words.

**Reservoir sampling**: In reservoir sampling, if we have a reservoir of size S:

* We store the first S elements of the stream
* When the n<sup>th</sup> element arrives (let's call it X<sub>n</sub>):
   * With probability 1 - s/n, we ignore this element.
   * With probability s/n, we:
      * Discard a random element from the reservoir
      * Add element X<sub>n</sub> to the reservoir (calling *add_to_reservoir*)
      
<font size="-1" color="gray">(Remove this cell when delivering.)</font>

## 1.1. Implement reservoir sampling <font color="#2680EB">[20%]</font>

Implement a function `add_reservoir(reservoir, item, max_size)` that adds an item to the reservoir, maintaining its size. If the reservoir is already of size *max_size*, a random item is selected and evicted *before* adding the item.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Implement add_to_reservoir**</font>

In [ ]:
def add_to_reservoir(reservoir, item, max_reservoir_size):
    
    # YOUR ANSWER HERE

    assert len(reservoir) <= max_reservoir_size, "Implementation error, reservoir is larger than max_reservoir_size"

Create a function to iterate through the file using the reservoir sampling method seen in class. In this function you will decide, for every item, whether to call *add_to_reservoir* or to ignore the item.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Implement reservoir_sampling**</font>

In [ ]:
def reservoir_sampling(filename, parts_of_speech, reservoir_size, max_words=-1, report_every=-1, use_pos_prefix=True):
    reservoir = []

    # YOUR ANSWER HERE

    return (words_seen, reservoir)

Test your function using the following code:

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE THIS CODE AS-IS

reservoir_size = 1500
(words_seen, reservoir) = reservoir_sampling(INPUT_FILE, [POS_NOUN, POS_ADJECTIVE, POS_VERB], reservoir_size, max_words=MAX_WORDS, use_pos_prefix=True)

print(f"Sampled {len(reservoir)} of {words_seen} words")

## 1.2. Estimate frequencies using reservoir sampling <font color="#2680EB">[30%]</font>

The reservoir contains repeated items. You can compute the absolute frequencies of the top 10 using the following code.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE THIS CODE AS-IS

freq = {}
for item in reservoir:
    freq[item] = reservoir.count(item)

most_frequent_items = sorted([(frequency, word) for word, frequency in freq.items()], reverse=True)[:20]
for absolute_frequency, word in most_frequent_items:
    print("%d %s" % (absolute_frequency, word))

Write code to compute the 30 most frequent words in the reservoir and their relative frequencies, as percentages.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Print the top 30 most frequent words in the reservoir.** Include their relative frequencies, as percentages in relation to the number of words read.</font>

In [ ]:
NUM_ITEMS = 30

# YOUR ANSWER HERE


If you see an item C times in the reservoir, you can estimate the item appears *C x total_word_count / reservoir_size* times in the entire dataset (*total_word_count* is the total number of words in the input). 

For various sizes of the reservoir, e.g., 10, 50, 100, 150, 200, 500, ..., list the top-5 **proper nouns** and your estimate of their frequency in the entire dataset.
 
<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS_IS

POS_PROPER_NOUN = "NNP"

def word_count(input_filename, pos=[]):

    # The instructor used co-pilot to generate this function
    count = 0
    with gzip.open(input_filename, "rt", encoding='utf8') as file:
        for line in file:
            if pos:
                for sentence in nltk.sent_tokenize(line):
                    tagged = nltk.pos_tag(nltk.word_tokenize(sentence))
                    count += len([word for word, tag in tagged if tag in pos])
            else:
                count += len(nltk.word_tokenize(line))
    return count

TOTAL_WORD_COUNT = word_count(INPUT_FILE)
TOTAL_PROPER_NOUN_COUNT = word_count(INPUT_FILE, pos=[POS_PROPER_NOUN])

print(f"The file contains {TOTAL_WORD_COUNT} words, of which {TOTAL_PROPER_NOUN_COUNT} are proper nouns")

<font size="+1" color="#2680EB">**Print the top 10 proper nouns in the dataset and their estimated frequency.** Try with various reservoir sizes.</font>

In [ ]:
NUM_ITEMS = 10

def print_top_proper_nouns(input_filename, reservoir_size, max_words=-1, report_every=-1):

    (words_seen, reservoir) = reservoir_sampling(input_filename, [POS_PROPER_NOUN], reservoir_size, max_words=max_words, report_every=report_every, use_pos_prefix=False)

    # YOUR ANSWER HERE

print_top_proper_nouns(INPUT_FILE, 1000)

<font size="+1" color="#2680EB">**Answer the questions below** regarding the variations observed with various reservoir sizes.</font>

**Answer.**

Which reservoir sizes did you test?

* YOUR ANSWER HERE

At which reservoir size do the top-3 proper nouns estabilize across runs? What are those proper nouns? Who they correspond to in the novel?

* YOUR ANSWER HERE

# 2. Determine approximately the distinct number of words <font color="#2680EB">[50%]</font>

We will estimate the number of distinct words without creating a dictionary or hash table, but instead, we will use the Flajolet-Martin probabilistic counting method.

**Flajolet-Martin probabilistic counting**:

* For several passes
   * Create hash funcion h
   * For every element *u* in the stream:
      * Compute hash value *h(u)*
      * Let *r(u)* be the number of trailing zeroes in *h(u)*
      * Maintain *R* as the maximum value of *r(u)* seen so far
   * Add *2<sup>R</sup>* as an estimate for the number of distinct elements *u* seen
* The final estimate is the average or the median of the estimates found in each pass

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

Use this function to count trailing zeroes in the binary representation of a number.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE THIS CODE AS-IS

def count_trailing_zeroes(number):
    count = 0
    while number & 1 == 0:
        count += 1
        number = number >> 1
    return count

Use this function to generate a random hash function. Note this generates a function, so you can do `hash_function = random_hash_function()` and then call `hash_function(x)` to compute the hash value of `x`. 

We want to make sure each hash is different, so we will create each hash function with a different [salt](https://en.wikipedia.org/wiki/Salt_(cryptography)), which is an additional input that we will take using a good random string generator from the [secrets](https://docs.python.org/3/library/secrets.html) library.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE THIS CODE AS-IS

def random_hash_function():
    # We use a cryptographically safe generator for the salt of our hash function
    salt = secrets.token_bytes(32)
    return lambda string: hash(string + str(salt))

Perform *number_of_passes* passes over the file, reading the entire file on each pass (we don't use the reservoir in this part). In each pass, create a new hash function and use it to hash userids. Keep the maximum number of trailing zeroes seen in the hash value of a userid. 

To count all words, leave the parts of speech empty, using ``read_by_parts_of_speech(INPUT_FILE, [], max_words=MAX_WORDS)``

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Implement a generator of estimates** that does a number of passes over the data.</font>

In [ ]:

NUMBER_OF_PASSES = 5

estimates = []

for i in range(NUMBER_OF_PASSES):
    hasher = random_hash_function()

    # YOUR ANSWER HERE

    estimates.append(estimation)
    print(f"Estimate on pass {i+1}: {estimation} distinct words")

In [ ]:
# Leave this code as-is

print("* Average of estimates: %.1f" % statistics.mean(estimates))
print("* Median  of estimates: %.1f" % statistics.median(estimates))

You can increase the limit of words to read, and perform more passes, but do not use more than 5 minutes of computing time.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Answer the questions below** regarding the probabilistic measurements you have done.</font>

**Answer.**

What was the maximum number of passes you did? How much time did it take?

* YOUR ANSWER HERE

Considering that the Odyssey actually has around 8,200 distinct words, how did the estimate based on **average** perform?

* YOUR ANSWER HERE

Considering that the Odyssey actually has around 8,200 distinct words, how did the estimate based on **median** perform?

* YOUR ANSWER HERE

Can you comment on both estimates?

* YOUR ANSWER HERE


# DELIVER (individually)

Remember to read the section on "delivering your code" in the [course evaluation guidelines](https://github.com/chatox/data-mining-course/blob/master/upf/upf-evaluation.md).

Deliver this notebook, renamed as `PSxx_Unnnnnn_YourName.ipynb` where:
* *PSxx* is the practice number.
* *Unnnnnn* is your UPF id number.
* *YourName* is your name.

## Extra points available

For more learning and extra points, pick one of these activities:

(1) Plot the average and median estimate for runs 1 to X, for various values of X.

(2) Notice that the number of **distinct** words in a corpus, as a function of the **total** number of words in the corpus, follows an empirical law known as [Heap's Law](https://en.wikipedia.org/wiki/Heaps%27_law).

Plot the estimate of distinct words versus number of words read. For this, you need to try with various values of `max_word`. Please note that using probabilistic counting means a substantial amount of noise will be introduced and perhaps the Heap's law will not be clear in your plot.

**Note:** if you go for the extra points, add ``<font size="+2" color="#2680EB">Additional results</font>`` at the top of your notebook. 

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+2" color="#00A365">I hereby declare that I completed this practice myself, that my answers were not written by an AI-enabled code assistant, and that except for the code provided by the course instructors, all of my code, report, and figures were produced by myself.</font>